# 08. 이미지 필터링

> 강의에서 작성한 `filter*.py`와 `cartooncamera.py`를 실행 순서대로 담았습니다.

강의 화면의 코드 순서와 파일명을 유지했습니다. 데이터 파일은 코드에 표시된 `./data` 또는 `../data` 상대 경로에 두세요.


## 사용자 커널과 평균 블러


In [ ]:
import sys
import cv2
import numpy as np

cap = cv2.VideoCapture('/dev/video0')

if not cap.isOpened():
    print("카메라를 열 수 없습니다.")
    sys.exit(1)

kernel = np.ones((5, 5), dtype=np.float32) / 25

while True:
    ret, frame = cap.read()

    if not ret:
        print("카메라 영상을 읽을 수 없습니다.")
        break

    # 사용자 정의 평균 필터
    dst1 = cv2.filter2D(frame, -1, kernel)

    # OpenCV 평균 블러
    dst2 = cv2.blur(frame, (5, 5))

    cv2.imshow('Original', frame)
    cv2.imshow('filter2D', dst1)
    cv2.imshow('blur', dst2)

    # ESC 키를 누르면 종료
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()


## 3×3·5×5·7×7 평균 필터


In [ ]:
import sys
import cv2

src = cv2.imread("./data/gorilla.jpg", cv2.IMREAD_GRAYSCALE)

if src is None:
    print("이미지를 읽을 수 없습니다.")
    sys.exit()

cv2.imshow("src", src)

for ksize in (3, 5, 7):
    dst = cv2.blur(src, (ksize, ksize))
    desc = "Mean: {}x{}".format(ksize, ksize)

    cv2.putText(
        dst,
        desc,
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.0,
        255,
        1,
        cv2.LINE_AA
    )

    cv2.imshow(desc, dst)

cv2.waitKey(0)
cv2.destroyAllWindows()


## 가우시안 필터


In [ ]:
import sys
import numpy as np
import cv2

src = cv2.imread('./data/gorilla.jpg', cv2.IMREAD_GRAYSCALE)

if src is None:
    print('Image load failed!')
    sys.exit()

cv2.imshow('src', src)

for sigma in range(1, 6):
    # sigma 값을 이용하여 가우시안 필터링
    dst = cv2.GaussianBlur(src, (0, 0), sigma)

    desc = 'sigma = {}'.format(sigma)
    cv2.putText(dst, desc, (10, 30), cv2.FONT_HERSHEY_SIMPLEX,
                1.0, 255, 1, cv2.LINE_AA)

    cv2.imshow(desc, dst)
    cv2.waitKey()

cv2.destroyAllWindows()


## 언샤프 마스크 샤프닝


In [ ]:
import sys
import numpy as np
import cv2

src = cv2.imread('./data/gorilla.jpg', cv2.IMREAD_GRAYSCALE)

if src is None:
    print('Image load failed!')
    sys.exit()

blr = cv2.GaussianBlur(src, (0, 0), 10)
dst = np.clip(2.0*src - blr, 0, 255).astype(np.uint8)

cv2.imshow('src', src)
cv2.imshow('blr', blr)
cv2.imshow('dst', dst)
cv2.waitKey()

cv2.destroyAllWindows()


## 메디안 필터 잡음 제거


In [ ]:
import sys
import cv2
import numpy as np
src = cv2.imread('./data/noise.bmp', cv2.IMREAD_GRAYSCALE)

if src is None:
    print('Image load failed!')
    sys.exit()

dst1= cv2.medianBlur(src, 3)
dst = cv2.medianBlur(src, 5)

blr = cv2.GaussianBlur(dst, (0, 0), 10)
dst_gaussian = np.clip(2.0*dst - blr, 0, 255).astype(np.uint8)

cv2.imshow('src', src)
cv2.imshow('dst1', dst1)
cv2.imshow('dst', dst)
cv2.imshow('dst_gaussian', dst_gaussian)
cv2.waitKey()

cv2.destroyAllWindows()


## 카툰·스케치 카메라


In [ ]:
import sys

import cv2
import numpy as np


def cartoon_filter(img):
    h, w = img.shape[:2]
    small = cv2.resize(img, (w // 2, h // 2))

    smooth = cv2.bilateralFilter(small, -1, 20, 7)
    edge = 255 - cv2.Canny(small, 80, 120)
    edge = cv2.cvtColor(edge, cv2.COLOR_GRAY2BGR)

    result = cv2.bitwise_and(smooth, edge)
    return cv2.resize(
        result,
        (w, h),
        interpolation=cv2.INTER_NEAREST,
    )


def pencil_sketch(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (0, 0), 3)
    sketch = cv2.divide(gray, blur, scale=255)

    return cv2.cvtColor(sketch, cv2.COLOR_GRAY2BGR)


def negative_filter(img):
    return cv2.bitwise_not(img)


def pixel_art_filter(img):
    h, w = img.shape[:2]

    pixel_size = 20
    small_w = max(1, w // pixel_size)
    small_h = max(1, h // pixel_size)

    small = cv2.resize(
        img,
        (small_w, small_h),
        interpolation=cv2.INTER_LINEAR,
    )

    return cv2.resize(
        small,
        (w, h),
        interpolation=cv2.INTER_NEAREST,
    )


def emboss_filter(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    kernel = np.array([
        [-2, -1, 0],
        [-1,  1, 1],
        [0,   1, 2],
    ])

    emboss = cv2.filter2D(gray, -1, kernel)
    emboss = cv2.add(emboss, 128)

    return cv2.cvtColor(emboss, cv2.COLOR_GRAY2BGR)


def thermal_filter(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return cv2.applyColorMap(gray, cv2.COLORMAP_JET)


FILTERS = [
    ("Normal", lambda frame: frame),
    ("Cartoon", cartoon_filter),
    ("Pencil Sketch", pencil_sketch),
    ("Negative", negative_filter),
    ("Pixel Art", pixel_art_filter),
    ("Emboss", emboss_filter),
    ("Thermal Camera", thermal_filter),
]


cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Video open failed!")
    sys.exit()

cam_mode = 0

while True:
    ret, frame = cap.read()

    if not ret:
        print("Frame capture failed!")
        break

    filter_name, filter_function = FILTERS[cam_mode]
    result = filter_function(frame)

    cv2.putText(
        result,
        filter_name,
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        2,
        cv2.LINE_AA,
    )

    cv2.imshow("Fun Filter Camera", result)

    key = cv2.waitKey(1) & 0xFF

    if key == 27 or key == ord("q"):
        break
    elif key == ord(" "):
        cam_mode = (cam_mode + 1) % len(FILTERS)

cap.release()
cv2.destroyAllWindows()
